<a href="https://colab.research.google.com/github/Fdez96/proyectofinaltokio/blob/Ejercicios-completos/ejercicio3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark


In [ ]:
# Inicializar la sesión de Spark
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName('AirTraffic').getOrCreate()

In [ ]:
# Cargar el archivo en un DataFrame
df = spark.read.csv("Air_Traffic_Passenger_Statistics.csv", header=True, inferSchema=True)

In [ ]:
num_companias = df.select("Operating Airline").distinct().count()
print(f"Número de compañias diferentes: {num_companias}")


Número de compañias diferentes: 77


In [ ]:
from pyspark.sql.functions import avg

media_pasajeros = df.groupBy("Operating Airline").agg(avg("Passenger Count").alias("media_pasajeros"))
media_pasajeros.show()


+--------------------+------------------+
|   Operating Airline|   media_pasajeros|
+--------------------+------------------+
|          Icelandair|            2799.7|
|         Ameriflight|               5.0|
|      Cathay Pacific|17121.325581395347|
|          Aeromexico| 5463.822222222222|
|      Etihad Airways| 6476.088235294118|
| Philippine Airlines|10248.635658914729|
|United Airlines -...| 48915.46750232126|
|    Turkish Airlines| 8162.416666666667|
| Swiss International| 6061.640287769784|
|    Independence Air|            6391.3|
|Miami Air Interna...|           107.375|
|          Air France|11589.077519379845|
|      Japan Airlines| 6470.332046332046|
|    Midwest Airlines|            3883.0|
|      Atlas Air, Inc|              34.0|
|    JetBlue Airways | 35261.13963963964|
|       China Eastern| 5498.402777777777|
|   Mexicana Airlines| 7993.806451612903|
|         Air Canada |18251.560109289618|
|       Allegiant Air|         1516.8125|
+--------------------+------------

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("GEO Region").orderBy(df["Passenger Count"].desc())

df_unicos = df.withColumn("row_num", row_number().over(window)) \
              .filter("row_num == 1") \
              .drop("row_num")


In [ ]:
# Guardar la media de pasajeros por compañía
media_pasajeros.coalesce(1).write.csv("media_pasajeros_por_compania.csv", header=True, mode='overwrite')

# Guardar los registros únicos por “GEO Región”
df_unicos.coalesce(1).write.csv("mayor_pasajeros_por_georegion.csv", header=True, mode='overwrite')